In [1]:

import os, math, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

TRAIN_NPZ_PATH = "/content/drive/MyDrive/quickdraw_train.npz"  # <-- change this
VAL_SIZE = 0.2
BATCH_SIZE = 512
EPOCHS = 40  # ≤ 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:

# Part A: "Pancake" = shallow but wide (≤ 3M params)
PANCAKE_HIDDEN = [1536, 1024]  # ~2.80M params including BN
# Part B: "Tower" = deep but narrow (≤ 3M params)
TOWER_HIDDEN   = [256, 256, 256, 256, 256, 256, 256]  # ~0.60M params including BN

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


In [5]:

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [6]:

class QuickDrawDataset(Dataset):
    def __init__(self, x, y=None):
        # Normalize to [0,1] and keep flat (784) because MLP. (28x28 images flattened)
        self.x = (x.astype(np.float32) / 255.0)
        self.y = y

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        xi = torch.from_numpy(self.x[idx])  # shape (784,)
        if self.y is not None:
            yi = torch.tensor(self.y[idx], dtype=torch.long)
            return xi, yi
        return xi


In [7]:

class MLP(nn.Module):
    def __init__(self, input_size: int, hidden: list, num_classes: int, dropout: float = 0.15, use_bn=True):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.GELU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x is already flat (N, 784)
        return self.net(x)

def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [8]:

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, preds_all, y_all = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds_all.extend(torch.argmax(logits, dim=1).detach().cpu().numpy())
        y_all.extend(y.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_all, preds_all)
    return avg_loss, acc


In [9]:

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, preds_all, y_all = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        preds_all.extend(torch.argmax(logits, dim=1).detach().cpu().numpy())
        y_all.extend(y.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_all, preds_all)
    return avg_loss, acc


In [10]:

def plot_curves(history, title, out_path):
    # history: list of dicts with keys: epoch, train_loss, train_acc, val_loss, val_acc
    epochs = [h["epoch"] for h in history]
    tr_loss = [h["train_loss"] for h in history]
    va_loss = [h["val_loss"] for h in history]
    tr_acc  = [h["train_acc"] for h in history]
    va_acc  = [h["val_acc"] for h in history]

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epochs, tr_loss, label="Train Loss")
    ax[0].plot(epochs, va_loss, label="Val Loss")
    ax[0].set_title(f"{title} - Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Loss")
    ax[0].legend()

    ax[1].plot(epochs, tr_acc, label="Train Acc")
    ax[1].plot(epochs, va_acc, label="Val Acc")
    ax[1].set_title(f"{title} - Accuracy")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Accuracy")
    ax[1].legend()

    fig.tight_layout()
    fig.savefig(out_path, dpi=140)
    plt.close(fig)


In [11]:

def run_experiment(name, hidden, x_train, y_train,
                   epochs=EPOCHS, batch_size=BATCH_SIZE,
                   lr=LEARNING_RATE, wd=WEIGHT_DECAY, dropout=0.15):
    # Split once into train/val (stratified)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
    tr_idx, va_idx = next(sss.split(x_train, y_train))
    x_tr, y_tr = x_train[tr_idx], y_train[tr_idx]
    x_va, y_va = x_train[va_idx], y_train[va_idx]

    ds_tr = QuickDrawDataset(x_tr, y_tr)
    ds_va = QuickDrawDataset(x_va, y_va)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = MLP(input_size=784, hidden=hidden, num_classes=15, dropout=dropout, use_bn=True).to(DEVICE)
    params = count_trainable_params(model)
    assert params <= 3_000_000, f"[{name}] Param budget exceeded: {params}"

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    # Mild cosine schedule; keeps things simple and stable
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    history = []
    best_val = -1.0
    best = {"train_acc": 0.0, "val_acc": 0.0, "epoch": 0}

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, dl_tr, optimizer, criterion)
        va_loss, va_acc = evaluate(model, dl_va, criterion)
        scheduler.step()

        history.append({
                    "epoch": ep,
                    "train_loss": tr_loss, "train_acc": tr_acc,
                    "val_loss": va_loss, "val_acc": va_acc,
                })

        if va_acc > best_val:
            best_val = va_acc
            best = {"train_acc": tr_acc, "val_acc": va_acc, "epoch": ep}

        if ep % 5 == 0 or ep == 1 or ep == epochs:
            print(f"[{name}] Epoch {ep:02d}/{epochs} | "
                f"Train Acc: {tr_acc:.4f} | Val Acc: {va_acc:.4f} | Params: {params:,}")

    # Save curves
    out_png = os.path.join(OUT_DIR, f"{name}_curves.png")
    plot_curves(history, title=name, out_path=out_png)

    return {
        "name": name,
        "params": params,
        "epochs": epochs,
        "best_epoch": best["epoch"],
        "best_train_acc": round(best["train_acc"], 4),
        "best_val_acc": round(best["val_acc"], 4),
        "curve_path": out_png
            }



In [12]:
# ---------- Main ----------
def main():
    # Load NPZ (expects keys: x_train, y_train, class_names)
    data = np.load(TRAIN_NPZ_PATH)
    X = data["x_train"]  # (N, 784) uint8
    y = data["y_train"]  # (N,) integers 0..14
    assert X.ndim == 2 and X.shape[1] == 784, "Expected flattened 28x28 images (N, 784)"
    assert y.ndim == 1, "Expected labels shape (N,)"

    print(f"Loaded: X={X.shape}, y={y.shape}. Classes={len(np.unique(y))}")

    # Part A (Pancake): shallow but wide
    res_A = run_experiment(
        name="partA_pancake",
        hidden=PANCAKE_HIDDEN,
        x_train=X,
        y_train=y,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        wd=WEIGHT_DECAY,
        dropout=0.15
    )

    # Part B (Tower): deep but narrow
    res_B = run_experiment(
        name="partB_tower",
        hidden=TOWER_HIDDEN,
        x_train=X,
        y_train=y,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        wd=WEIGHT_DECAY,
        dropout=0.10
    )

    # Save a CSV-like results file for the report
    csv_path = os.path.join(OUT_DIR, "partA_B_results.csv")
    with open(csv_path, "w") as f:
        f.write("Model,Params,Epochs,BestEpoch,BestTrainAcc,BestValAcc,Curves\n")
        for r in [res_A, res_B]:
            f.write(f"{r['name']},{r['params']},{r['epochs']},{r['best_epoch']},"
                    f"{r['best_train_acc']},{r['best_val_acc']},{r['curve_path']}\n")

    print("\n===== RESULTS (copy into report) =====")
    print("Model, Params, Epochs, BestEpoch, BestTrainAcc, BestValAcc")
    for r in [res_A, res_B]:
        print(f"{r['name']}, {r['params']:,}, {r['epochs']}, {r['best_epoch']}, "
              f"{r['best_train_acc']:.4f}, {r['best_val_acc']:.4f}")
    print(f"\nSaved curves and table to: {OUT_DIR}")


In [13]:

if __name__ == "__main__":
    main()

Loaded: X=(60000, 784), y=(60000,). Classes=15
[partA_pancake] Epoch 01/40 | Train Acc: 0.6746 | Val Acc: 0.7311 | Params: 2,800,143
[partA_pancake] Epoch 05/40 | Train Acc: 0.8832 | Val Acc: 0.7689 | Params: 2,800,143
[partA_pancake] Epoch 10/40 | Train Acc: 0.9778 | Val Acc: 0.7798 | Params: 2,800,143
[partA_pancake] Epoch 15/40 | Train Acc: 0.9974 | Val Acc: 0.7866 | Params: 2,800,143
[partA_pancake] Epoch 20/40 | Train Acc: 0.9995 | Val Acc: 0.7911 | Params: 2,800,143
[partA_pancake] Epoch 25/40 | Train Acc: 0.9999 | Val Acc: 0.7937 | Params: 2,800,143
[partA_pancake] Epoch 30/40 | Train Acc: 1.0000 | Val Acc: 0.7940 | Params: 2,800,143
[partA_pancake] Epoch 35/40 | Train Acc: 1.0000 | Val Acc: 0.7951 | Params: 2,800,143
[partA_pancake] Epoch 40/40 | Train Acc: 1.0000 | Val Acc: 0.7947 | Params: 2,800,143
[partB_tower] Epoch 01/40 | Train Acc: 0.5927 | Val Acc: 0.7040 | Params: 603,151
[partB_tower] Epoch 05/40 | Train Acc: 0.7874 | Val Acc: 0.7531 | Params: 603,151
[partB_tower] E

In [14]:
!cp -r /content/outputs /content/drive/MyDrive/AI600_PA2_Part_AB_outputs